In [ ]:
import sys; sys.path.append('/Users/arnavshah/Code/dnaBLT/training/data/iterators')
from training.data.iterators.v_args import TrainArgs
from training.data.iterators.v_arrow_iterator import ArrowFileIterator
from training.data.iterators.v_preprocess_iterator import PreprocessIterator

train_args = TrainArgs()
# train_args.data.buffer_size = 64
dataloader = train_args.data.build_from_rank(0, 1, 0, 1, "train")

ZeroDivisionError: integer modulo by zero

In [18]:
import torch
import numpy as np
from tqdm import trange

all_nonzero_patch_lengths = []
length_sum = 0
patch_sum = 0

for _ in trange(1000):
    big_batch = next(preprocess_iterator)
    patch_lengths = big_batch.patch_lengths
    length_sum += (patch_lengths != 0).sum()
    patch_sum += patch_lengths.sum()
    # Flatten, filter, convert to numpy, and append
    nonzero_patch_lengths = patch_lengths[patch_lengths != 0]
    all_nonzero_patch_lengths.append(nonzero_patch_lengths.cpu())  # ensure on CPU if tensor

print("Average patch size", patch_sum / length_sum)
# Concatenate all batches into a single tensor
all_nonzero_patch_lengths = torch.cat(all_nonzero_patch_lengths, dim=0)

# Now proceed with your logic
tensor_np = all_nonzero_patch_lengths.numpy()
max_patch = all_nonzero_patch_lengths.max().item()
value_range = np.arange(1, max_patch + 1)
counts = np.bincount(tensor_np, minlength=max_patch + 1)[1:]  # skip index 0
weighted = counts * value_range
cdf = weighted.cumsum() / weighted.sum()
idx_99 = (cdf > 0.99).argmax()

print(f"99% of the weighted patch length mass is covered by patch length: {value_range[idx_99]}")

100%|██████████| 1000/1000 [03:50<00:00,  4.34it/s]


Average patch size tensor(2.1598)
99% of the weighted patch length mass is covered by patch length: 250
